# Calidad, Limpieza y Preparación de Datos
## Proyecto Integrador — Minería de Datos I

**Integrantes:** Valeria Martinetti y Santiago Gallardo

**Dataset:** `streaming_users_dirty.json` — Usuarios de plataforma de streaming

---

### Objetivo de este notebook

Aplicar transformaciones justificadas sobre el dataset original para obtener
un dataset limpio y analíticamente válido, listo para ser utilizado en las
etapas de EDA y PCA.

Cada decisión tomada en este notebook está respaldada por evidencia observada
durante la inspección inicial (`01_inspección_inicial.ipynb`).

### Criterio de trabajo

- El dataset original (`data/raw/streaming_users_dirty.json`) no será modificado.
- Todas las transformaciones se aplican sobre una copia de trabajo.
- Cada paso queda registrado en `logs/pipeline_log.csv`.
- El dataset resultante se guarda en `data/processed/`.


## 1 Setup Inicial

Preparación de las herramientas y archivo con el que se va a trabajar.

In [84]:
import pandas as pd
import numpy as np

df_raw = pd.read_json('../data/raw/streaming_users_dirty.json')
df = df_raw.copy()

Se crea el contenedor del log, y se define la función "registrar" para ir almacenando los cambios. Luego inicializamos el log

In [85]:
log = []

def registrar(df, paso, descripcion):
    log.append({
        'Paso': paso,
        'Descripción': descripcion,
        'Filas': len(df),
        'Nulos': int(df.isnull().sum().sum()),
        'Retención (%)': round(len(df) / len(df_raw) * 100, 2)
    })

registrar(df, 1, 'Preparación')

## 2 Eliminación de Duplicados
Se eliminan los elementos que tienen los mismos valores en todas las columnas

En la exploración de datos encontramos que teníamos 160 datos duplicados en users_id, pero cuando aplicamos drop_duplicates() solo elimina 126...

In [86]:
cant_antes = len(df)
df = df.drop_duplicates()

print(f"Dataset con duplicados {len(df_raw)}")
print(f"Dataset sin duplicados {len(df)}")

registrar(df, 2, "Eliminación de Duplicados")

Dataset con duplicados 8160
Dataset sin duplicados 8034


## 3 Normalización
Se agrupan las columnas que tinen los mismos valores escritos de diferentes formas agrupandolas bajo un mismo valor

In [87]:
mapa_plan = {
    'Básico': 'Básico', 'basico': 'Básico', 'BASICO': 'Básico',
    'Basic': 'Básico', 'básico': 'Básico',
    'Estándar': 'Estándar', 'Std': 'Estándar', 'Estándar ': 'Estándar',
    'estandar': 'Estándar', 'STANDARD': 'Estándar',
    'Premium': 'Premium', 'Premium ': 'Premium', 'PREMIUM': 'Premium',
    'Premiun': 'Premium', 'premium': 'Premium',
}

df['subscription_plan'] = df['subscription_plan'].map(mapa_plan)
df['subscription_plan'].value_counts()


subscription_plan
Básico      3609
Estándar    2833
Premium     1592
Name: count, dtype: int64

In [88]:
mapa_pais = {
    'Brasil': 'Brasil', 'brasil': 'Brasil', 'Brazil': 'Brasil', 'BRA': 'Brasil',
    'Colombia': 'Colombia', 'colombia': 'Colombia', 'COL': 'Colombia',
    'Chile': 'Chile', 'chile': 'Chile', 'CHL': 'Chile',
    'México': 'México', 'méxico': 'México', 'Mexico': 'México', 'MEX': 'México',
    'Uruguay': 'Uruguay', 'uruguay': 'Uruguay', 'URY': 'Uruguay',
    'Perú': 'Perú', 'perú': 'Perú', 'Peru': 'Perú', 'PER': 'Perú',
    'Argentina': 'Argentina', 'argentina': 'Argentina', 'ARG': 'Argentina',
}

df['country'] = df['country'].map(mapa_pais)
df['country'].value_counts()


country
Brasil       1164
México       1162
Chile        1151
Uruguay      1146
Colombia     1145
Perú         1139
Argentina    1101
Name: count, dtype: int64

In [89]:
mapa_genero = {
    'Comedia': 'Comedia', 'COMEDIA': 'Comedia', 'comedy': 'Comedia',
    'Drama': 'Drama', 'DRAMA': 'Drama', 'drama': 'Drama',
    'Documental': 'Documental', 'documental': 'Documental', 'Documentary': 'Documental', 'DOC': 'Documental',
    'Thriller': 'Thriller', 'THRILLER': 'Thriller', 'thriler': 'Thriller',
    'Romance': 'Romance', 'ROMANCE': 'Romance', 'romance': 'Romance',
    'Acción': 'Acción', 'ACCIÓN': 'Acción', 'accion': 'Acción', 'Action': 'Acción',
    'Crimen': 'Crime', 'CRIME': 'Crime', 'crime': 'Crime', 'Crimen': 'Crime',
}

df['favorite_genre'] = df['favorite_genre'].map(mapa_genero)
print(df['favorite_genre'].value_counts())
print(df.isnull().sum())

registrar(df, 3, "Normalización")


favorite_genre
Comedia       1126
Drama         1114
Documental    1111
Acción        1110
Thriller      1097
Romance       1096
Crime           40
Name: count, dtype: int64
user_id                        0
age                            0
subscription_plan              0
monthly_watch_time_mins      193
country                       26
favorite_genre              1340
last_login_date              320
customer_support_tickets       0
dtype: int64


## 4 Tratamiento de Outliers
Se apartaron los valores físicamente imposibles reemplazándolos por NaN para su posterior imputación.

In [90]:
edad_invalida = ~df['age'].between(1, 100)
df.loc[edad_invalida, 'age'] = np.nan
print(f"edades inválidas → NaN: {edad_invalida.sum()}")

tiempo_invalido = df['monthly_watch_time_mins'].notna() & ~df['monthly_watch_time_mins'].between(0, 44640)
df.loc[tiempo_invalido, 'monthly_watch_time_mins'] = np.nan
print(f"tiempo de visualización inválidos → NaN: {tiempo_invalido.sum()}")

tickets_invalido = (df['customer_support_tickets'] < 0) | (df['customer_support_tickets'] > 20)
df.loc[tickets_invalido, 'customer_support_tickets'] = np.nan
print(f"tickets inválidos → NaN: {tickets_invalido.sum()}")

registrar(df, 4, "Tratamiento de Outliers")

edades inválidas → NaN: 98
tiempo de visualización inválidos → NaN: 80
tickets inválidos → NaN: 96


In [91]:
# Definimos una función para aplicar winsorización (IQR con k=3)
def winsorizar_k3(df_work, columna):
    Q1 = df_work[columna].quantile(0.25)
    Q3 = df_work[columna].quantile(0.75)
    IQR = Q3 - Q1
    
    # Límite superior extremo (k=3)
    limite_sup = Q3 + 3.0 * IQR
    
    # Contamos cuántos valores serán modificados para tener registro
    outliers = (df_work[columna] > limite_sup).sum()
    print(f"{columna} -> {outliers} outliers topeados en el valor: {limite_sup:.2f}")
    
    # Limitamos los valores sin alterar el mínimo (ya que no hay valores irreales bajos aquí)
    df_work[columna] = df_work[columna].clip(upper=limite_sup)
    
    return df_work

# Aplicamos la función a nuestras variables
df = winsorizar_k3(df, 'monthly_watch_time_mins')
df = winsorizar_k3(df, 'customer_support_tickets')

# Registramos el paso en el log de auditoría
registrar(df, 4.5, "Winsorización (k=3) de extremos reales en tiempo y tickets")


monthly_watch_time_mins -> 108 outliers topeados en el valor: 2693.40
customer_support_tickets -> 14 outliers topeados en el valor: 4.00


## 5 Parseo de fechas
Se ajustaron los valores de fechas al formato en el que la máquina pueda entenderlos

In [92]:
fecha_antes = df['last_login_date'].isnull().sum()

df['last_login_date'] = pd.to_datetime(df['last_login_date'], errors='coerce')

fecha_despues = df['last_login_date'].isnull().sum()

print(f"NaN antes del parseo:  {fecha_antes}")
print(f"NaN después del parseo: {fecha_despues}")
print(f"Fechas inválidas nuevas → NaN: {fecha_despues - fecha_antes}")

registrar(df, 5, "Parseo de fechas")

NaN antes del parseo:  320
NaN después del parseo: 769
Fechas inválidas nuevas → NaN: 449


## 6 Imputación de valores nulos
Se tratan a los nulos según la naturaleza de cada columna

In [93]:
for x in ['age', 'monthly_watch_time_mins', 'customer_support_tickets']:
    mediana = df[x].median()
    nulos = df[x].isnull().sum()
    df[x] = df[x].fillna(mediana)
    print(f"{x}: {nulos} nulos imputados con mediana = {mediana:.2f}")

print("="*50)

nulos_genero = df['favorite_genre'].isnull().sum()
df['favorite_genre'] = df['favorite_genre'].fillna('Otros')
print(f"favorite_genre: {nulos_genero} nulos imputados con 'Otros'")

print("="*40)

nulos_fecha = df['last_login_date'].isnull().sum()
print(nulos_fecha)

print("="*30)

print(f"\nNulos totales restantes: {df.isnull().sum().sum()}")
print(f"Shape final: {df.shape}")

registrar(df, 6, "Imputación de valores nulos")

age: 98 nulos imputados con mediana = 33.00
monthly_watch_time_mins: 273 nulos imputados con mediana = 758.50
customer_support_tickets: 96 nulos imputados con mediana = 1.00
favorite_genre: 1340 nulos imputados con 'Otros'
769

Nulos totales restantes: 795
Shape final: (8034, 8)


In [94]:
pd.DataFrame(log)
df.to_csv('../data/processed/streaming_users_clean.csv', index=False)

# Log del pipeline
pd.DataFrame(log).to_csv('../logs/pipeline_log.csv', index=False)
